# Arabic Written Dialect Scan with MARBERTv2
This notebook is configured for a full run by default using `IbrahimAmin/marbertv2-arabic-written-dialect-classifier`.

Main outputs:
- `predictions.jsonl`: one record per processed row
- `row_probabilities.jsonl`: per-row probabilities for later threshold sweeps
- `dialect_progress.log`: visible live progress log in the run folder
- `progress_latest.json`: visible latest machine-readable status in the run folder
- `.logs/dialect_progress.log`: legacy hidden compatibility copy
- `.logs/progress_latest.json`: legacy hidden compatibility copy
- `summary.json`: latest/final summary


In [1]:
import json
from collections import deque
from pathlib import Path

import pandas as pd

In [2]:
# Full-run config. This notebook is not in smoke mode by default.

PYTHON_BIN = '/home/MohammadNabulsi/whisper/.venv/bin/python'
SCRIPT_PATH = Path('/home/MohammadNabulsi/whisper/arabic_text_dialect_scan_marbertv2_written.py')

DATA_ROOTS = [
    '/home/MohammadNabulsi/whisper/data/masc_c_only/data',
    '/home/MohammadNabulsi/whisper/processed_qasr_segments/train',
]

OUTPUT_DIR = '/home/MohammadNabulsi/whisper/Runs/text_dialect_scan_marbertv2_written'
MODEL_ID = 'IbrahimAmin/marbertv2-arabic-written-dialect-classifier'

UNKNOWN_THRESHOLD = 0.55
TARGET_LABEL = 'LEV'
TARGET_THRESHOLD = 0.40
MAX_LENGTH = 512
MAX_SAMPLES = None
DEVICE = 'auto'
BATCH_SIZE = 256
LOG_EVERY = 250
RESUME = True


In [3]:
cmd = [
    PYTHON_BIN, str(SCRIPT_PATH),
    '--output-dir', OUTPUT_DIR,
    '--model-id', MODEL_ID,
    '--device', DEVICE,
    '--unknown-threshold', str(UNKNOWN_THRESHOLD),
    '--target-label', TARGET_LABEL,
    '--target-threshold', str(TARGET_THRESHOLD),
    '--max-length', str(MAX_LENGTH),
    '--batch-size', str(BATCH_SIZE),
    '--log-every', str(LOG_EVERY),
]

for root in DATA_ROOTS:
    cmd += ['--data-root', root]

if MAX_SAMPLES is not None:
    cmd += ['--max-samples', str(MAX_SAMPLES)]

if RESUME:
    cmd += ['--resume']

print(' '.join(cmd))

/home/MohammadNabulsi/whisper/.venv/bin/python /home/MohammadNabulsi/whisper/arabic_text_dialect_scan_marbertv2_written.py --output-dir /home/MohammadNabulsi/whisper/Runs/text_dialect_scan_marbertv2_written --model-id IbrahimAmin/marbertv2-arabic-written-dialect-classifier --device auto --unknown-threshold 0.55 --target-label LEV --target-threshold 0.4 --max-length 512 --batch-size 256 --log-every 250 --data-root /home/MohammadNabulsi/whisper/data/masc_c_only/data --data-root /home/MohammadNabulsi/whisper/processed_qasr_segments/train --resume


In [4]:
# Run the full scan.
!{' '.join(cmd)}

Found 721 shards.
 - /home/MohammadNabulsi/whisper/data/masc_c_only/data/test-00000-of-00009.parquet
 - /home/MohammadNabulsi/whisper/data/masc_c_only/data/test-00001-of-00009.parquet
 - /home/MohammadNabulsi/whisper/data/masc_c_only/data/test-00002-of-00009.parquet
 - /home/MohammadNabulsi/whisper/data/masc_c_only/data/test-00003-of-00009.parquet
 - /home/MohammadNabulsi/whisper/data/masc_c_only/data/test-00004-of-00009.parquet
 - /home/MohammadNabulsi/whisper/data/masc_c_only/data/test-00005-of-00009.parquet
 - /home/MohammadNabulsi/whisper/data/masc_c_only/data/test-00006-of-00009.parquet
 - /home/MohammadNabulsi/whisper/data/masc_c_only/data/test-00007-of-00009.parquet
 - /home/MohammadNabulsi/whisper/data/masc_c_only/data/test-00008-of-00009.parquet
 - /home/MohammadNabulsi/whisper/data/masc_c_only/data/train-00000-of-00400.parquet
 ... plus 711 more
Counting total rows across shards for overall progress tracking...
Total rows discovered: 1563477
Visible progress log: /home/Mohamm

In [5]:
latest_path = Path(OUTPUT_DIR) / 'progress_latest.json'
legacy_latest_path = Path(OUTPUT_DIR) / '.logs' / 'progress_latest.json'
chosen_path = latest_path if latest_path.exists() else legacy_latest_path
if chosen_path.exists():
    summary = json.loads(chosen_path.read_text(encoding='utf-8'))
    summary['status_path'] = str(chosen_path)
else:
    summary = {
        'status': 'missing_progress',
        'message': 'Run the scan cell successfully first. progress_latest.json has not been written yet.',
        'expected_path': str(latest_path),
        'legacy_expected_path': str(legacy_latest_path),
    }
summary

{'time': '2026-06-26 16:14:51',
 'attempted_in_this_run': 1491362,
 'completed_total': 1505239,
 'completed_before_run': 13877,
 'total_rows': 1563477,
 'overall_progress_percentage': 96.2751,
 'processed_ok_total': 1505239,
 'skipped_total': 0,
 'errors_total': 0,
 'current_file': 'DONE',
 'current_uid': 'DONE',
 'elapsed_seconds': 1503.48,
 'rows_per_second': 991.94,
 'estimated_remaining_seconds': 58.71,
 'estimated_remaining_human': '58s',
 'dialect_counts': {'MAGHREB': 41296,
  'LEV': 370925,
  'MSA': 862285,
  'GLF': 67880,
  'EGY': 56465,
  'unseen': 106388},
 'dialect_percentages': {'MAGHREB': 2.7435,
  'LEV': 24.6423,
  'MSA': 57.2856,
  'GLF': 4.5096,
  'EGY': 3.7512,
  'unseen': 7.0678},
 'unseen_count': 106388,
 'unseen_percentage': 7.0678,
 'status_path': '/home/MohammadNabulsi/whisper/Runs/text_dialect_scan_marbertv2_written/progress_latest.json'}

In [6]:
if 'dialect_percentages' in summary:
    pct = summary['dialect_percentages']
    counts = summary['dialect_counts']
    pd.DataFrame([
        {'label': label, 'count': counts[label], 'percentage': pct[label]}
        for label in counts
    ]).sort_values('percentage', ascending=False)
else:
    pd.DataFrame()

In [7]:
prob_path = Path(OUTPUT_DIR) / 'row_probabilities.jsonl'

if prob_path.exists():
    tail = deque(maxlen=5)
    with prob_path.open('r', encoding='utf-8') as f:
        for line in f:
            tail.append(json.loads(line))
    preview = []
    for rec in tail:
        preview.append({
            'uid': rec['uid'],
            'top_label': rec['top_label'],
            'top_score': rec['top_score'],
            'final_label': rec['final_label'],
            'target_label_score': rec.get('target_label_score'),
            'text': rec['text'][:160],
        })
    pd.DataFrame(preview)
else:
    pd.DataFrame()

In [8]:
# Threshold experiment for Levantine candidates without rerunning inference.

ANALYSIS_LABEL = 'LEV'
ANALYSIS_THRESHOLD = 0.40
TOP_K = 20

prob_path = Path(OUTPUT_DIR) / 'row_probabilities.jsonl'
matches = []
total_scored = 0

if prob_path.exists():
    with prob_path.open('r', encoding='utf-8') as f:
        for line in f:
            rec = json.loads(line)
            score = rec.get('label_scores', {}).get(ANALYSIS_LABEL)
            if score is None:
                continue
            total_scored += 1
            if score >= ANALYSIS_THRESHOLD:
                matches.append({
                    'uid': rec['uid'],
                    'score': score,
                    'top_label': rec['top_label'],
                    'final_label': rec['final_label'],
                    'text': rec['text'],
                })

match_df = pd.DataFrame(matches).sort_values('score', ascending=False).head(TOP_K) if matches else pd.DataFrame()
print({
    'analysis_label': ANALYSIS_LABEL,
    'analysis_threshold': ANALYSIS_THRESHOLD,
    'rows_scored': total_scored,
    'rows_meeting_threshold': len(matches),
})
match_df

{'analysis_label': 'LEV', 'analysis_threshold': 0.4, 'rows_scored': 1505239, 'rows_meeting_threshold': 416944}


,uid,score,top_label,final_label,text
35117,train-00212-of-00400.parquet:video_id=3Yrh0K6L-HY,0.999452,LEV,LEV,وكنت راجعه على بيتي وكنت عم بسمع صوت ام ماسكه ...
141776,shard_000067.arrow:uid=qasr:2778BDE4-10DC-433F...,0.999448,LEV,LEV,حسيت حاجه بقلبي قالت لي ايه قرصتني بقلبي هيك م...
68784,shard_000002.arrow:uid=qasr:02214A9B-272A-4BE1...,0.999446,LEV,LEV,فقعد ربع ساعه ورجع،
209855,shard_000127.arrow:uid=qasr:4DBD0D76-3106-47EF...,0.999446,LEV,LEV,طلعنا علي هون توقعنا انه تعيين جلسه ويلا،
40477,train-00245-of-00400.parquet:video_id=dEofWxaL7e8,0.999445,LEV,LEV,عشان هيك كانت الحياه صعبه كتير
321363,shard_000224.arrow:uid=qasr:8D4351F0-D290-4A6F...,0.999444,LEV,LEV,المره الماضيه عم تقلي ست اشهر ما في سلاح،
2497,train-00014-of-00400.parquet:video_id=3Yrh0K6L-HY,0.999444,LEV,LEV,وكانوا الجيران عم يسقوا الزريعه وريحه التراب ط...
43825,train-00265-of-00400.parquet:video_id=3Yrh0K6L-HY,0.999443,LEV,LEV,وقبل ما اوصل على البيت لقيت انه عم بشم ريحه ال...
56793,train-00348-of-00400.parquet:video_id=6mBjw08qVYg,0.999443,LEV,LEV,بس ما مان بدو يعمل هالمحاوله مع ادم بشكل مباشر
330435,shard_000232.arrow:uid=qasr:919BD94D-14FC-47E9...,0.999442,LEV,LEV,ماما يارا انا عرفتِني تعمل لي هيك تهز براسها ب...


In [9]:
log_path = Path(OUTPUT_DIR) / 'dialect_progress.log'
legacy_log_path = Path(OUTPUT_DIR) / '.logs' / 'dialect_progress.log'
chosen_log_path = log_path if log_path.exists() else legacy_log_path
print('Showing log from:', chosen_log_path)
!tail -n 20 "{chosen_log_path}"

Showing log from: /home/MohammadNabulsi/whisper/Runs/text_dialect_scan_marbertv2_written/dialect_progress.log
[2026-06-26 16:14:47] progress=95.97% completed=1500529/1563477 ok=1500529 skipped=0 errors=0 rate=991.41 rows/s eta=1m03s unseen=7.07% MAGHREB=2.75% LEV=24.63% MSA=57.28% GLF=4.51% EGY=3.76% unseen=7.07% file=/home/MohammadNabulsi/whisper/processed_qasr_segments/train/shard_000306.arrow uid=shard_000306.arrow:uid=qasr:C00A1D54-5F0C-46BC-A83A-9755DA294AD3:C00A1D54_5F0C_46BC_A83A_9755DA294AD3_utt_117_align
[2026-06-26 16:14:49] progress=96.00% completed=1500878/1563477 ok=1500878 skipped=0 errors=0 rate=990.18 rows/s eta=1m03s unseen=7.07% MAGHREB=2.75% LEV=24.63% MSA=57.29% GLF=4.51% EGY=3.75% unseen=7.07% file=/home/MohammadNabulsi/whisper/processed_qasr_segments/train/shard_000307.arrow uid=shard_000307.arrow:uid=qasr:C02041AD-B439-441E-82D1-2B88DB4D23E4:C02041AD_B439_441E_82D1_2B88DB4D23E4_utt_177_align
[2026-06-26 16:14:49] progress=96.01% completed=1501134/1563477 ok=15011